In [0]:
CREATE OR REPLACE TABLE telecom_lakehouse.bronze.customer_payments_raw
USING DELTA
COMMENT 'Synthetic raw telecom customer payment transactions'
AS
WITH payment_invoices AS (
    SELECT
        invoice_id,
        customer_id,
        invoice_date,
        paid_amount,
        CASE
            WHEN paid_amount = 0 THEN 0
            WHEN PMOD(ROW_NUMBER() OVER (ORDER BY invoice_id), 3) = 0
                THEN 1
            ELSE 2
        END AS payment_count
    FROM telecom_lakehouse.bronze.billing_invoices_raw
    WHERE paid_amount > 0
),
payment_rows AS (
    SELECT
        invoice_id,
        customer_id,
        invoice_date,
        paid_amount,
        payment_count,
        EXPLODE(
            CASE
                WHEN payment_count = 1 THEN ARRAY(1)
                ELSE ARRAY(1, 2)
            END
        ) AS payment_sequence
    FROM payment_invoices
)
SELECT
    CONCAT(
        'PAY',
        LPAD(
            CAST(ROW_NUMBER() OVER (
                ORDER BY invoice_id, payment_sequence
            ) AS STRING),
            9,
            '0'
        )
    ) AS payment_id,

    invoice_id,
    customer_id,

    DATE_ADD(
        invoice_date,
        CAST(PMOD(ROW_NUMBER() OVER (
            ORDER BY invoice_id, payment_sequence
        ) * 3, 25) AS INT)
    ) AS payment_date,

    ROUND(
        paid_amount / payment_count,
        2
    ) AS payment_amount,

    CASE
        WHEN PMOD(ROW_NUMBER() OVER (
            ORDER BY invoice_id, payment_sequence
        ), 4) = 0 THEN 'UPI'
        WHEN PMOD(ROW_NUMBER() OVER (
            ORDER BY invoice_id, payment_sequence
        ), 4) = 1 THEN 'CARD'
        WHEN PMOD(ROW_NUMBER() OVER (
            ORDER BY invoice_id, payment_sequence
        ), 4) = 2 THEN 'NET_BANKING'
        ELSE 'WALLET'
    END AS payment_method,

    'SUCCESS' AS payment_status,
    'PAYMENT_GATEWAY' AS source_system,
    'BATCH_001' AS ingest_batch_id,
    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM payment_rows;